In [ ]:
zip_path="/mnt/2TB_WD/rishi/cnemc_data.zip"
unzip_dir="/mnt/2TB_WD/rishi/cnemc_data"
import zipfile, os

os.makedirs(unzip_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(unzip_dir)

In [1]:
import glob
import os
import re
import pandas as pd
from tqdm import tqdm

data_dir = "/mnt/2TB_WD/rishi/cnemc_data/cnemc_data"
out_dir  = "/mnt/2TB_WD/rishi/cnemc_data/by_site_pollutant"
os.makedirs(out_dir, exist_ok=True)

start_date = "20220101"
end_date   = "20251231"

keep_types = {"CO", "NO2", "O3", "PM10", "PM2.5", "SO2"}
pollutant_col = {
    "CO":    "CO (mg/m³)",
    "NO2":   "NO2 (µg/m³)",
    "O3":    "Ozone (µg/m³)",
    "PM10":  "PM10 (µg/m³)",
    "PM2.5": "PM2.5 (µg/m³)",
    "SO2":   "SO2 (µg/m³)",
}
pollutant_name = {k: v.split(" ")[0] for k, v in pollutant_col.items()}
pollutant_name["O3"] = "Ozone"

full_index = pd.date_range(
    start=pd.to_datetime(start_date, format="%Y%m%d"),
    end=pd.to_datetime(end_date, format="%Y%m%d") + pd.Timedelta(hours=23),
    freq="h",
    name="Timestamp",
)

def file_date(f):
    m = re.search(r"(\d{8})", os.path.basename(f))
    return m.group(1) if m else ""

all_files = sorted(glob.glob(f"{data_dir}/*.csv"))
day_files = [f for f in all_files if start_date <= file_date(f) <= end_date]
print(f"Reading {len(day_files)} files into RAM...")

big_df = pd.concat(
    [pd.read_csv(f) for f in tqdm(day_files)],
    ignore_index=True,
)
big_df = big_df[big_df["type"].isin(keep_types)]
big_df["Timestamp"] = pd.to_datetime(big_df["date"].astype(str), format="%Y%m%d") + pd.to_timedelta(big_df["hour"], unit="h")
big_df = big_df.drop(columns=["date", "hour"])
print(f"Loaded. Shape: {big_df.shape}")

meta_cols = {"type", "Timestamp"}
sites = sorted(set(big_df.columns) - meta_cols)
print(f"Found {len(sites)} unique sites")

for site in tqdm(sites, desc="sites"):
    sub = big_df[["Timestamp", "type", site]].dropna(subset=[site])
    for pollutant, group in sub.groupby("type"):
        col = pollutant_col[pollutant]
        out = (
            group[["Timestamp", site]]
            .rename(columns={site: col})
            .set_index("Timestamp")
            .reindex(full_index)
        )
        out_path = os.path.join(out_dir, f"site_{site}_{pollutant_name[pollutant]}.csv")
        out.to_csv(out_path)

Reading 1461 files into RAM...


100%|██████████| 1461/1461 [01:18<00:00, 18.59it/s]


Loaded. Shape: (208289, 2030)
Found 2028 unique sites


sites: 100%|██████████| 2028/2028 [11:34<00:00,  2.92it/s]
